<a href="https://colab.research.google.com/github/anjalikrishnaak6-research/neuroimag_EEG/blob/main/EEG_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mne
!pip install pandas
!pip install h5py

In [2]:
!pip install --upgrade mne

In [3]:
import h5py
print(h5py.__version__)

3.15.1


In [4]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

In [5]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os

folder_path = "/content/drive/MyDrive/EEG_datasets/1/"

print(os.listdir(folder_path))

['eeg_VEP_visual_oddball_session_1_task_Oddball_task_subjectLabId_01_B_01_VEP_recording_1.set', 'event_VEP_visual_oddball_session_1_task_Oddball_task_subjectLabId_01_recording_1.tsv', 'eeg_VEP_visual_oddball_session_10_task_Oddball_task_subjectLabId_10_B_10_VEP_recording_1.set', 'event_VEP_visual_oddball_session_10_task_Oddball_task_subjectLabId_10_recording_1.tsv', 'event_VEP_visual_oddball_session_11_task_Oddball_task_subjectLabId_11_recording_1.tsv', 'eeg_VEP_visual_oddball_session_11_task_Oddball_task_subjectLabId_11_B_11_VEP_recording_1.set', 'eeg_VEP_visual_oddball_session_12_task_Oddball_task_subjectLabId_12_B_12_VEP_recording_1.set', 'event_VEP_visual_oddball_session_12_task_Oddball_task_subjectLabId_12_recording_1.tsv', 'eeg_VEP_visual_oddball_session_13_task_Oddball_task_subjectLabId_13_B_13_VEP_recording_1.set', 'event_VEP_visual_oddball_session_13_task_Oddball_task_subjectLabId_13_recording_1.tsv', 'eeg_VEP_visual_oddball_session_14_task_Oddball_task_subjectLabId_14_B_14_VE

In [10]:
import mne
import h5py

set_path = "/content/drive/MyDrive/EEG_datasets/1/eeg_VEP_visual_oddball_session_1_task_Oddball_task_subjectLabId_01_B_01_VEP_recording_1.set"

# Now that we've confirmed h5py can open it and packages are up to date,
# we can directly use mne.io.read_raw_eeglab
raw = mne.io.read_raw_eeglab(set_path, preload=True)

print(raw)

NotImplementedError: Please use HDF reader for matlab v7.3 files, e.g. h5py

In [9]:
# basic preprocessing
raw.filter(0.1, 30., fir_design='firwin')  #bandpass filter

raw.notch_filter(freqs=50)  # change to 60 if needed

raw.set_eeg_reference('average', projection=False)

NameError: name 'raw' is not defined

In [ ]:
ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter='auto')
ica.fit(raw)

ica.plot_components()
ica.plot_sources(raw)

In [ ]:
ica.exclude = [0, 2]

raw = ica.apply(raw)

In [ ]:
events, event_id = mne.events_from_annotations(raw)

print(event_id)

In [ ]:
event_dict = {
    'standard': 1,
    'target': 2
}

In [ ]:
# epoching -200 to 800ms

epochs = mne.Epochs(
    raw,
    events,
    event_id=event_dict,
    tmin=-0.2,
    tmax=0.8,
    baseline=(-0.2, 0),
    preload=True
)

In [ ]:
# sepatate target and standard
epochs_target = epochs['target']
epochs_standard = epochs['standard']

In [ ]:
# verification of P3 existence
evoked_target = epochs_target.average()
evoked_standard = epochs_standard.average()

evoked_target.plot()
evoked_standard.plot()

In [ ]:
# prepare deep learning tensors

X = epochs.get_data()  # shape: trials × channels × timepoints
y = epochs.events[:, -1]  # event labels

# Convert labels to binary 0/1
label_map = {event_dict['standard']: 0, event_dict['target']: 1}
y = np.array([label_map[label] for label in y])

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# save the tensor

np.save("/content/drive/MyDrive/X_eeg.npy", X)
np.save("/content/drive/MyDrive/y_labels.npy", y)

In [ ]:
# extract the classical p3 fetaures

times = epochs.times
p3_window = np.where((times >= 0.3) & (times <= 0.6))[0]

# Define electrodes of interest
roi_channels = ['Cz', 'CPz', 'Pz', 'CP1', 'CP2', 'P3', 'P4']
roi_indices = [epochs.ch_names.index(ch) for ch in roi_channels if ch in epochs.ch_names]

X_data = epochs.get_data()

p3_mean_amp = X_data[:, roi_indices][:, :, p3_window].mean(axis=(1,2))
p3_peak_amp = X_data[:, roi_indices][:, :, p3_window].max(axis=(1,2))

# Peak latency
p3_peak_latency = []
for trial in X_data[:, roi_indices][:, :, p3_window]:
    mean_roi = trial.mean(axis=0)
    peak_idx = np.argmax(mean_roi)
    latency = times[p3_window][peak_idx]
    p3_peak_latency.append(latency)

p3_peak_latency = np.array(p3_peak_latency)

# Feature matrix
X_classical = np.column_stack([p3_mean_amp, p3_peak_amp, p3_peak_latency])

print("Classical feature shape:", X_classical.shape)

In [ ]:
np.save("/content/drive/MyDrive/X_classical.npy", X_classical)